# Debug Your Broken RAG With Kayak

This is the main internal notebook for the course sequence.

Treat it like a debugging session, not like a lecture.

Start from one user problem:

- my RAG system still cannot reliably find the evidence I need

Then move in the natural order most users actually follow:

1. poke the failure and see what changes
2. compare against the simpler baselines people already use
3. name the failure class only after the behavior is visible
4. only then talk about performance and deployment shape

The other internal notebooks remain supporting labs. This one is the main story.

Backstage verification note: its deterministic teaching spine is mechanically covered by `python.tests.test_course_sequence_smoke`.
The real judged-slice bridge in Section 3 additionally expects the cached `LIMIT-small` task.

## What You Should Walk Away Knowing

After playing through this notebook, the learner should be able to look at a miss and ask:

- did compressed scoring hide the evidence?
- is the retrieval unit itself wrong?
- did the shortlist drop the oracle before reranking even started?

They should also be able to compare those failures against two baselines many teams actually use:

- one vector per whole document
- one vector per chunk


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "python" / "kayak").exists():
            return candidate
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "python"))

import kayak
from kayak_bridge.judged_metrics import summarize_ranked_task

CACHE_ROOT = REPO_ROOT / ".cache" / "kayak"
print("Working from repo root:", REPO_ROOT)
print("Backends available here:", kayak.available_backends())

## 1. First, Get One Honest Reference Point

Start with one small miss where the behavior is easy to see.
The relevant document contains both evidence tokens, but it also carries a lot of noise.
A distractor repeats only part of the query.

If the winner changes when you remove the simplification, that tells you something real.


In [ ]:
DIM = 64
TOKEN_TO_INDEX: dict[str, int] = {}


def token_vector(token: str) -> np.ndarray:
    index = TOKEN_TO_INDEX.setdefault(token, len(TOKEN_TO_INDEX))
    vector = np.zeros(DIM, dtype=np.float32)
    vector[index] = np.float32(1.0)
    return vector


def encode_tokens(tokens: list[str]) -> np.ndarray:
    return np.stack([token_vector(token) for token in tokens])


def mean_vec(matrix: np.ndarray) -> np.ndarray:
    return np.mean(matrix, axis=0, dtype=np.float32, keepdims=True).astype(np.float32)


def build_chunked_index_from_rows(rows: dict[str, list[str]], chunk_size: int) -> tuple[kayak.LateIndex, dict[str, str]]:
    chunk_ids = []
    chunk_vectors = []
    parent_by_chunk: dict[str, str] = {}
    for doc_id, tokens in rows.items():
        matrix = encode_tokens(tokens)
        for chunk_index, start in enumerate(range(0, len(matrix), chunk_size)):
            chunk_id = f"{doc_id}::chunk{chunk_index}"
            chunk_ids.append(chunk_id)
            chunk_vectors.append(mean_vec(matrix[start : start + chunk_size]))
            parent_by_chunk[chunk_id] = doc_id
    return kayak.documents(chunk_ids, chunk_vectors).pack(), parent_by_chunk


def build_chunked_index_from_task(task: dict, chunk_size: int) -> tuple[kayak.LateIndex, dict[str, str]]:
    chunk_ids = []
    chunk_vectors = []
    parent_by_chunk: dict[str, str] = {}
    for row in task['documents']:
        matrix = np.asarray(row['vectors'], dtype=np.float32)
        for chunk_index, start in enumerate(range(0, len(matrix), chunk_size)):
            chunk_id = f"{row['doc_id']}::chunk{chunk_index}"
            chunk_ids.append(chunk_id)
            chunk_vectors.append(mean_vec(matrix[start : start + chunk_size]))
            parent_by_chunk[chunk_id] = row['doc_id']
    return kayak.documents(chunk_ids, chunk_vectors).pack(), parent_by_chunk


def dedup_parent_docs(chunk_hits, parent_by_chunk: dict[str, str], k: int) -> tuple[str, ...]:
    ranked_doc_ids = []
    seen = set()
    for hit in chunk_hits:
        doc_id = parent_by_chunk[hit.doc_id]
        if doc_id in seen:
            continue
        seen.add(doc_id)
        ranked_doc_ids.append(doc_id)
        if len(ranked_doc_ids) >= k:
            break
    return tuple(ranked_doc_ids)


In [ ]:
query_tokens = ["cancel", "subscription"]
documents = {
    "doc-relevant": query_tokens + [f"noise-{i}" for i in range(20)],
    "doc-partial": ["cancel", "cancel", "cancel", "cancel"],
    "doc-other": ["billing", "invoice"],
}

exact_index = kayak.documents(
    list(documents.keys()),
    [encode_tokens(tokens) for tokens in documents.values()],
).pack()
exact_query = kayak.query(encode_tokens(query_tokens))
onevec_index = kayak.documents(
    list(documents.keys()),
    [mean_vec(encode_tokens(tokens)) for tokens in documents.values()],
).pack()
onevec_query = kayak.query(mean_vec(encode_tokens(query_tokens)))

exact_hits = kayak.search(exact_query, exact_index, k=3, backend=kayak.NUMPY_REFERENCE_BACKEND)
onevec_hits = kayak.search(onevec_query, onevec_index, k=3, backend=kayak.NUMPY_REFERENCE_BACKEND)

print("If I keep full token interaction:", [(hit.doc_id, hit.score) for hit in exact_hits])
print("If I collapse to one vector:", [(hit.doc_id, hit.score) for hit in onevec_hits])

## 2. Then Ask If The Retrieval Unit Is Wrong

This is the moment where many people say, "we changed chunking and everything moved."
Sometimes that helps because one useful evidence pocket gets isolated.
Sometimes it hurts because the answer only makes sense when several pieces stay together.


In [ ]:
rows = {
    "relevant": ["noise1", "noise2", "noise3", "noise4", "cancel", "subscription", "noise5", "noise6"],
    "partial": ["cancel", "noise7", "cancel", "noise8"],
    "other": ["invoice", "billing"],
}
query = kayak.query(mean_vec(encode_tokens(["cancel", "subscription"])))
onevec_index = kayak.documents(list(rows), [mean_vec(encode_tokens(tokens)) for tokens in rows.values()]).pack()
chunk_index, parent_by_chunk = build_chunked_index_from_rows(rows, chunk_size=2)

print("If I keep one vector per document:", [(hit.doc_id, hit.score) for hit in kayak.search(query, onevec_index, k=3)])
chunk_hits = kayak.search(query, chunk_index, k=chunk_index.document_count)
print("If I chunk first and then dedupe docs:", dedup_parent_docs(chunk_hits, parent_by_chunk, 3))

In [ ]:
rows = {
    "relevant": ["alpha", "noise1", "beta", "noise2", "gamma", "noise3"],
    "partial": ["alpha", "beta", "noise4", "noise5"],
    "other": ["invoice", "billing"],
}
exact_query = kayak.query(encode_tokens(["alpha", "beta", "gamma"]))
exact_index = kayak.documents(list(rows), [encode_tokens(tokens) for tokens in rows.values()]).pack()
onevec_query = kayak.query(mean_vec(encode_tokens(["alpha", "beta", "gamma"])))
chunk_index, parent_by_chunk = build_chunked_index_from_rows(rows, chunk_size=2)

print("If I keep the full document together:", [(hit.doc_id, hit.score) for hit in kayak.search(exact_query, exact_index, k=3)])
chunk_hits = kayak.search(onevec_query, chunk_index, k=chunk_index.document_count)
print("If I chunk first and then dedupe docs:", dedup_parent_docs(chunk_hits, parent_by_chunk, 3))

## 3. Move To A Real Judged Slice

Now stop playing only with toys and check the same instinct on labeled data.
The default slice here is `LIMIT-small` because it is small enough to inspect, already cached locally, and adversarial enough to make single-vector shortcuts visible.


In [ ]:
limit_path = CACHE_ROOT / "limit_small_real_subset" / "python_task.json"
limit_task = json.loads(limit_path.read_text())
limit_index = kayak.documents(
    [row["doc_id"] for row in limit_task["documents"]],
    [np.asarray(row["vectors"], dtype=np.float32) for row in limit_task["documents"]],
    texts=[row["text"] for row in limit_task["documents"]],
).pack()
limit_onevec_index = kayak.documents(
    [row["doc_id"] for row in limit_task["documents"]],
    [mean_vec(np.asarray(row["vectors"], dtype=np.float32)) for row in limit_task["documents"]],
    texts=[row["text"] for row in limit_task["documents"]],
).pack()
limit_chunk16_index, limit_parent16 = build_chunked_index_from_task(limit_task, 16)


In [ ]:
exact_ranked = []
onevec_ranked = []
chunk16_ranked = []
proxy_ranked = []

for row in limit_task['queries']:
    query_matrix = np.asarray(row['vectors'], dtype=np.float32)
    exact_query = kayak.query(query_matrix, text=row['text'])
    onevec_query = kayak.query(mean_vec(query_matrix), text=row['text'])

    exact_hits = kayak.search(exact_query, limit_index, k=limit_task['k'], backend=kayak.NUMPY_REFERENCE_BACKEND)
    onevec_hits = kayak.search(onevec_query, limit_onevec_index, k=limit_task['k'], backend=kayak.NUMPY_REFERENCE_BACKEND)
    chunk16_hits = kayak.search(onevec_query, limit_chunk16_index, k=limit_chunk16_index.document_count, backend=kayak.NUMPY_REFERENCE_BACKEND)
    proxy_hits = kayak.search_with_plan(
        exact_query,
        limit_index,
        kayak.document_proxy_search_plan(final_k=limit_task['k'], candidate_k=limit_task['k']),
        backend=kayak.NUMPY_REFERENCE_BACKEND,
    ).hits

    exact_ranked.append(tuple(hit.doc_id for hit in exact_hits))
    onevec_ranked.append(tuple(hit.doc_id for hit in onevec_hits))
    chunk16_ranked.append(dedup_parent_docs(chunk16_hits, limit_parent16, limit_task['k']))
    proxy_ranked.append(tuple(hit.doc_id for hit in proxy_hits))

print('On LIMIT-small, exact late interaction gets:', summarize_ranked_task(task=limit_task, ranked_doc_ids_by_query=exact_ranked).primary_value)
print('On LIMIT-small, one vector per document gets:', summarize_ranked_task(task=limit_task, ranked_doc_ids_by_query=onevec_ranked).primary_value)
print('On LIMIT-small, chunk size 16 gets:', summarize_ranked_task(task=limit_task, ranked_doc_ids_by_query=chunk16_ranked).primary_value)
print('On LIMIT-small, the proxy path gets:', summarize_ranked_task(task=limit_task, ranked_doc_ids_by_query=proxy_ranked).primary_value)

In [ ]:
doc_by_id = {row['doc_id']: row for row in limit_task['documents']}
interesting_query = limit_task['queries'][0]
query_matrix = np.asarray(interesting_query['vectors'], dtype=np.float32)
exact_query = kayak.query(query_matrix, text=interesting_query['text'])
onevec_query = kayak.query(mean_vec(query_matrix), text=interesting_query['text'])

exact_hits = kayak.search(exact_query, limit_index, k=limit_task['k'], backend=kayak.NUMPY_REFERENCE_BACKEND)
onevec_hits = kayak.search(onevec_query, limit_onevec_index, k=limit_task['k'], backend=kayak.NUMPY_REFERENCE_BACKEND)
proxy_hits = kayak.search_with_plan(
    exact_query,
    limit_index,
    kayak.document_proxy_search_plan(final_k=limit_task['k'], candidate_k=limit_task['k']),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
).hits

print('Question to inspect:', interesting_query['text'])
print('Judged relevant docs:', interesting_query['relevant_doc_ids'])
print('Top-5 if I keep exact late interaction:', tuple(hit.doc_id for hit in exact_hits[:5]))
print('Top-5 if I collapse to one vector:', tuple(hit.doc_id for hit in onevec_hits[:5]))
print('Top-5 from the proxy path:', tuple(hit.doc_id for hit in proxy_hits[:5]))

## 4. Then Ask If The Shortlist Is The Real Bottleneck

This is where many teams blame the reranker when the real problem happened earlier.
A reranker cannot save a document that never entered the candidate window.


In [ ]:
query = kayak.query(encode_tokens(["cancel", "subscription"]))
index = kayak.documents(
    ["doc-partial", "doc-relevant", "doc-other"],
    [
        encode_tokens(["cancel", "cancel", "cancel"]),
        encode_tokens(["cancel", "subscription"]),
        encode_tokens(["invoice"]),
    ],
).pack()

exact_result = kayak.search_with_plan(
    query,
    index,
    kayak.exact_full_scan_search_plan(final_k=1, candidate_k=3),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)
narrow_result = kayak.search_with_plan(
    query,
    index,
    kayak.document_proxy_search_plan(final_k=1, candidate_k=1, query_vector_budget=1, document_vector_budget=1),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)
wider_result = kayak.search_with_plan(
    query,
    index,
    kayak.document_proxy_search_plan(final_k=1, candidate_k=2, query_vector_budget=1, document_vector_budget=1),
    backend=kayak.NUMPY_REFERENCE_BACKEND,
)

print('If I keep exact search, the top hit is:', [hit.doc_id for hit in exact_result.hits])
print('With a too-narrow candidate window, stage 1 keeps:', narrow_result.candidate_stage.candidate_doc_ids)
print('So the final hit becomes:', [hit.doc_id for hit in narrow_result.hits])
print('If I widen stage 1 a little, the candidates become:', wider_result.candidate_stage.candidate_doc_ids)
print('And the final hit recovers to:', [hit.doc_id for hit in wider_result.hits])

## 5. Only Now Talk About Performance And Integration

Once the learner knows where quality is being lost, the next practical questions are finally worth asking:

- should I reuse one loaded index?
- should I batch repeated queries?
- do I actually need to replace my store?

That is the point where the course should hand off to the supporting notes and the public storage/search docs.


## Final Diagnostic Checklist

If the learner remembers only one loop, it should be this one:

1. Can exact late interaction on the intended retrieval unit find the right document?
2. If not, is the retrieval unit wrong or is the evidence split across chunks?
3. If exact works, how far do one-vector and chunked one-vector baselines fall behind?
4. If exact works but the staged pipeline does not, is the shortlist dropping the oracle?
5. Only after those answers are clear should you optimize throughput or storage handoff.

That is the main course message:

- use Kayak to identify which simplification is hurting retrieval before you optimize around that simplification
